# ASTR 457: Foundations of Data Science in Astronomy

**Fall 2026, University of Illinois Urbana-Champaign**

Prof. Gautham Narayan | TA: Abha Vishwakarma

Thu Nov 19, 2026 - Day 23: Neural networks: from a neuron to survey pipelines

<img src="images/uiuc_logo.png" alt="University of Illinois Urbana-Champaign wordmark" width="220" style="display:block;margin:0 auto;">

In [ ]:
# Presenter setup (a RISE "skip" cell: it runs, but is not a slide). About 30 s: it trains every network on today's slides.
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.exceptions import ConvergenceWarning
from ipywidgets import interactive_output, SelectionSlider, HBox, VBox, Layout
from IPython.display import display
warnings.filterwarnings('ignore', category=ConvergenceWarning)
plt.rcParams.update({'font.size': 13, 'figure.figsize': (9.4, 3.6)})   # 9.4 in fits 1280 px; 7.4 in beside widget controls

import threading                                    # lock matplotlib's math-text parser (slider redraws run on another thread)
try:
    from matplotlib import _mathtext
    if not getattr(_mathtext.Parser.parse, '_locked', False):
        _mathtext_lock = threading.Lock()
        _unlocked_parse = _mathtext.Parser.parse
        def _locked_parse(self, *args, **kwargs):
            with _mathtext_lock:
                return _unlocked_parse(self, *args, **kwargs)
        _locked_parse._locked = True
        _mathtext.Parser.parse = _locked_parse
except (ImportError, AttributeError):
    print("could not lock matplotlib's math-text parser: do not move a slider while another cell is running")

def controls_right(fn, **widgets):
    for w in widgets.values():
        w.layout = Layout(width='230px')
        w.style = {'description_width': '95px'}
        w.continuous_update = False                 # redraw once, when the slider is released
    display(HBox([interactive_output(fn, widgets), VBox(list(widgets.values()), layout=Layout(justify_content='center', width='240px'))]))

# ---- SDSS DR8 main-sample galaxies with spectra: 20,000 drawn at random (instructor build script prep_photoz.py)
gal = pd.read_csv('data/day23_sdss_photoz.csv')
z_spec = gal.z_spec.values
gr = (gal.g - gal.r).values
X_all = np.c_[gal.u - gal.g, gal.g - gal.r, gal.r - gal.i, gal.i - gal.z, gal.r]   # four colors and one magnitude
FEATS = ['u-g', 'g-r', 'r-i', 'i-z', 'r']
X_tr, X_te, y_tr, y_te = train_test_split(X_all, z_spec, test_size=0.25, random_state=0)   # the test set is scored once, at the end
X_fit, X_val, y_fit, y_val = train_test_split(X_tr, y_tr, test_size=0.2, random_state=1)  # validation: carved from training, for choosing
scaler = StandardScaler().fit(X_tr)
A_tr, A_te = scaler.transform(X_tr), scaler.transform(X_te)
A_fit, A_val = scaler.transform(X_fit), scaler.transform(X_val)

def scores(z_phot, z_true):                         # the photo-z scores every survey quotes
    dz = (z_phot - z_true) / (1 + z_true)
    return {k: round(float(v), 4) for k, v in dict(scatter=1.4826 * np.median(np.abs(dz - np.median(dz))), outliers=np.mean(np.abs(dz) > 0.05), offset=np.median(dz)).items()}

models = {'straight line': LinearRegression(),
          'nearest neighbors (15)': KNeighborsRegressor(15),
          'random forest': RandomForestRegressor(200, min_samples_leaf=5, n_jobs=-1, random_state=0),
          'network (32, 32)': MLPRegressor(hidden_layer_sizes=(32, 32), early_stopping=True, max_iter=300, random_state=0)}
pred = {k: m.fit(A_tr, y_tr).predict(A_te) for k, m in models.items()}
table = pd.DataFrame({k: scores(p, y_te) for k, p in pred.items()}).T.round(4)
mlp = models['network (32, 32)']
N_PARAM = sum(w.size for w in mlp.coefs_) + sum(b.size for b in mlp.intercepts_)

# ---- overfitting: a big network (2 x 256) on 400 galaxies, one full pass (epoch) at a time; watched on the validation set
rng = np.random.default_rng(1)
i400 = rng.choice(len(A_fit), 400, replace=False)
big = MLPRegressor(hidden_layer_sizes=(256, 256), alpha=0, learning_rate_init=3e-4, batch_size=400, random_state=0)
EPOCHS, curve_tr, curve_va = np.arange(0, 1500, 10), [], []
rms = lambda p, t: np.sqrt(np.mean((p - t) ** 2))
for ep in range(1500):
    big.partial_fit(A_fit[i400], y_fit[i400])
    if ep % 10 == 0:
        curve_tr.append(rms(big.predict(A_fit[i400]), y_fit[i400])); curve_va.append(rms(big.predict(A_val), y_val))
BIG_PARAM = sum(w.size for w in big.coefs_) + sum(b.size for b in big.intercepts_)

# ---- width and weight decay on 20 galaxies, one feature (g-r): all eight fits trained once here, so the slider is instant
i20 = rng.choice(len(gr), 20, replace=False)
rest = np.setdiff1d(np.arange(len(gr)), i20)
std1 = lambda x: ((x - 0.75) / 0.2)[:, None]
GRID = np.linspace(0.2, 1.6, 400)
WIDTHS, ALPHAS = [20, 500], [1e-6, 3e-3, 1e-2, 3e-2]
fits20 = {}
for h in WIDTHS:
    for a in ALPHAS:
        m = MLPRegressor(hidden_layer_sizes=(h, h), solver='lbfgs', alpha=a, max_iter=50000, max_fun=100000, tol=0, random_state=0).fit(std1(gr[i20]), z_spec[i20])
        fits20[h, a] = (m.predict(std1(GRID)), rms(m.predict(std1(gr[i20])), z_spec[i20]), rms(m.predict(std1(gr[rest])), z_spec[rest]))

# ---- training on nearby galaxies only, then asking about distant ones
near = y_tr < 0.15
far = y_te > 0.2
shift = {k: m.fit(A_tr[near], y_tr[near]).predict(A_te[far]) for k, m in
         [('random forest', RandomForestRegressor(200, min_samples_leaf=5, n_jobs=-1, random_state=0)),
          ('network (32, 32)', MLPRegressor(hidden_layer_sizes=(32, 32), early_stopping=True, max_iter=300, random_state=0))]}

# ---- XOR: four groups, two classes, no straight line separates them
xr = np.random.default_rng(3)
XOR_X = np.vstack([xr.normal(c, 0.35, (60, 2)) for c in [(-1, -1), (1, 1), (-1, 1), (1, -1)]])
XOR_y = np.r_[np.zeros(120), np.ones(120)]
xor_line = LogisticRegression().fit(XOR_X, XOR_y)
xor_net = MLPClassifier(hidden_layer_sizes=(4,), activation='tanh', solver='lbfgs', max_iter=5000, random_state=0).fit(XOR_X, XOR_y)

print(f'{len(gal):,} galaxies; {len(y_tr):,} train, {len(y_te):,} test; the (32, 32) network has {N_PARAM:,} weights and biases')
print(table)
print('the (256, 256) network has', f'{BIG_PARAM:,}', 'weights and biases')


In [ ]:
# Plotting functions (a RISE "skip" cell). Every function here draws one slide's figure.

def plot_ladder():                                  # last Thursday's closing figure, re-drawn
    fig, ax = plt.subplots(figsize=(9.4, 2.4))
    Y, V = '#d9a400', '#7a4fbf'
    rows = [("Day 21 forest", '84 features, chosen by hand', 'the mapping from features to type'),
            ("Day 22 forest", '10 features, chosen by hand', 'the mapping from features to type'),
            ('today: neural networks', 'the network and the labels', 'the features themselves')]
    for i, (name, spec, learn) in enumerate(rows):
        y = 2 - i
        ax.text(0.0, y, name, ha='left', va='center', fontsize=11)
        ax.add_patch(plt.Rectangle((0.30, y - 0.35), 0.30, 0.7, color=Y, alpha=0.85))
        ax.text(0.45, y, spec, ha='center', va='center', fontsize=10)
        ax.add_patch(plt.Rectangle((0.62, y - 0.35), 0.38, 0.7, color=V, alpha=0.85))
        ax.text(0.81, y, learn, ha='center', va='center', fontsize=10, color='w')
    ax.text(0.45, 2.75, 'we choose', ha='center', fontsize=11, color=Y, weight='bold')
    ax.text(0.81, 2.75, 'the method learns', ha='center', fontsize=11, color=V, weight='bold')
    ax.set(xlim=(0, 1), ylim=(-0.5, 3.1)); ax.axis('off')
    plt.tight_layout(); plt.show()

def plot_activations():
    x = np.linspace(-4, 4, 400)
    fig, axs = plt.subplots(1, 4, figsize=(9.4, 2.3), sharex=True)
    for a, (name, f) in zip(axs, [('step (perceptron, 1958)', np.where(x > 0, 1.0, 0.0)), ('sigmoid', 1 / (1 + np.exp(-x))),
                                  ('tanh', np.tanh(x)), ('ReLU', np.maximum(x, 0))]):
        a.plot(x, f, 'C0', lw=2); a.axhline(0, color='0.7', lw=0.8); a.axvline(0, color='0.7', lw=0.8)
        a.set_title(name, fontsize=12); a.set_xlabel(r'$s = w \cdot x + b$', fontsize=11)
    axs[0].set_ylabel('output $g(s)$', fontsize=11)
    plt.tight_layout(); plt.show()

def plot_xor():
    gx, gy = np.meshgrid(np.linspace(-2.5, 2.5, 200), np.linspace(-2.5, 2.5, 200))
    G = np.c_[gx.ravel(), gy.ravel()]
    fig, axs = plt.subplots(1, 2, figsize=(9.4, 2.6))
    for a, (name, m) in zip(axs, [('one neuron (logistic regression)', xor_line), ('4 hidden neurons, then one output', xor_net)]):
        a.contourf(gx, gy, m.predict_proba(G)[:, 1].reshape(gx.shape), levels=[0, 0.5, 1], colors=['#cfe0f3', '#f6d5b8'])
        a.plot(*XOR_X[XOR_y == 0].T, 'o', color='C0', ms=4, label='class 0'); a.plot(*XOR_X[XOR_y == 1].T, 's', color='C1', ms=4, label='class 1')
        a.set_title(f'{name}: {m.score(XOR_X, XOR_y):.1%} right', fontsize=11); a.set(xticks=[], yticks=[])
    axs[0].legend(fontsize=9, loc='upper left', framealpha=0.9)
    plt.tight_layout(); plt.show()

def plot_kinks(n=0):                                # each ReLU adds one kink; enough kinks follow any smooth curve
    edges = np.linspace(0.3, 1.4, 23); mid = 0.5 * (edges[1:] + edges[:-1])
    med = np.array([np.median(z_spec[(gr > lo) & (gr < hi)]) for lo, hi in zip(edges[:-1], edges[1:])])
    knots = np.linspace(0.3, 1.4, n + 2)[1:-1]
    D = lambda x: np.c_[np.ones_like(x), x, *[np.maximum(x - k, 0) for k in knots]]
    coef = np.linalg.lstsq(D(mid), med, rcond=None)[0]
    resid = med - D(mid) @ coef
    xg = np.linspace(0.3, 1.4, 300)
    fig, axs = plt.subplots(2, 1, figsize=(7.4, 3.4), sharex=True, gridspec_kw={'height_ratios': [2, 1]})
    axs[0].plot(gr[:4000], z_spec[:4000], ',', color='0.75')
    axs[0].plot(mid, med, 'ko', ms=5, label='median redshift in each color bin')
    axs[0].plot(xg, D(xg) @ coef, 'C3', lw=2, label=f'a line plus {n} ReLU' + ('' if n == 1 else 's'))
    for a in axs:
        for j, k in enumerate(knots): a.axvline(k, color='C3', ls=':', lw=0.8, label='kink positions (we chose them)' if (a is axs[0] and j == 0) else None)
    axs[0].set(ylabel='redshift', xlim=(0.3, 1.4), ylim=(0, 0.3))
    axs[0].legend(fontsize=10, loc='upper left')
    axs[1].axhline(0, color='0.5', lw=0.8)
    axs[1].plot(mid, resid, 'ko-', ms=4)
    axs[1].set(xlabel='g - r color', ylabel='median - fit', ylim=(-0.045, 0.045))
    axs[1].set_title(f'rms residual {np.sqrt(np.mean(resid ** 2)):.4f}', fontsize=10)
    plt.tight_layout(); plt.show()

def kinks_controls():
    controls_right(plot_kinks, n=SelectionSlider(options=[0, 1, 4, 16], value=0, description='ReLUs'))

def plot_descent(lr=0.55, inputs='raw'):          # gradient descent on the straight-line fit z = c0 + c1 * (g - r), raw or scaled
    x = gr if inputs == 'raw' else (gr - gr.mean()) / gr.std()
    y = z_spec
    loss = lambda c0, c1: np.mean((y - c0 - c1 * x) ** 2)
    best = loss(*np.polyfit(x, y, 1)[::-1])                 # the exact least-squares answer (Day 9), for comparison
    c0, c1, path = 0.0, 0.0, [(0.0, 0.0)]
    for _ in range(30):
        r = y - c0 - c1 * x
        c0, c1 = c0 + lr * 2 * r.mean(), c1 + lr * 2 * (r * x).mean()
        path.append((c0, c1))
        if abs(c0) > 3 or abs(c1) > 3: break
    path = np.array(path)
    excess = np.array([loss(*q) for q in path]) - best
    C0, C1 = np.meshgrid(np.linspace(-0.15, 0.25, 120), np.linspace(-0.1, 0.3, 120))
    L = np.mean((y[::20, None, None] - C0 - C1 * x[::20, None, None]) ** 2, axis=0)
    fig, axs = plt.subplots(1, 2, figsize=(7.4, 2.6))
    axs[0].contour(C0, C1, np.log10(L), levels=15, cmap='viridis')
    axs[0].plot(path[:, 0], path[:, 1], 'o-', color='C3', ms=3)
    axs[0].set(xlabel='intercept $c_0$', ylabel='slope $c_1$', xlim=(-0.15, 0.25), ylim=(-0.1, 0.3))
    axs[0].set_title(f'{inputs} g - r: contours = loss, red = steps', fontsize=10)
    axs[1].semilogy(np.clip(excess, 1e-9, None), 'C3o-', ms=3)
    axs[1].set(xlabel='step', ylabel='loss - best possible', xlim=(-1, 31), ylim=(1e-9, 10))
    blew_up = excess[-1] > excess[0]
    axs[1].set_title('BLEW UP' if blew_up else (f'after {len(path) - 1} steps: {excess[-1]:.1e} above the best' if excess[-1] > 1e-9 else f'at the best fit after {int(np.argmax(excess < 1e-9))} steps'),
                     fontsize=10, color='C3' if blew_up else 'k')
    plt.tight_layout(); plt.show()

def descent_controls():
    controls_right(plot_descent, lr=SelectionSlider(options=[0.05, 0.55, 0.65, 1.05], value=0.55, description='learning rate'),
                   inputs=SelectionSlider(options=['raw', 'scaled'], value='raw', description='g - r input'))

def plot_photoz_data():
    fig, axs = plt.subplots(1, 2, figsize=(9.4, 3.1), gridspec_kw={'width_ratios': [1.3, 1]})
    s = axs[0].scatter(gal['g'][:6000] - gal['r'][:6000], gal['r'][:6000] - gal['i'][:6000], c=z_spec[:6000], s=2, cmap='plasma', vmin=0, vmax=0.3)
    plt.colorbar(s, ax=axs[0], label='redshift')
    axs[0].set(xlabel='g - r', ylabel='r - i', xlim=(0.2, 1.6), ylim=(0.1, 0.7))
    axs[1].hist(z_spec, bins=60, color='0.4')
    axs[1].set(xlabel='spectroscopic redshift', ylabel='galaxies')
    plt.tight_layout(); plt.show()

def plot_photoz_compare():
    fig, axs = plt.subplots(1, 3, figsize=(9.4, 3.0), sharex=True, sharey=True)
    for a, k in zip(axs, ['straight line', 'random forest', 'network (32, 32)']):
        a.plot(y_te, pred[k], ',', color='C0', alpha=0.5)
        a.plot([0, 0.4], [0, 0.4], 'k--', lw=0.8)
        a.set_title(f"{k}\nscatter {table.loc[k, 'scatter']:.3f}, outliers {table.loc[k, 'outliers']:.1%}", fontsize=10)
        a.set(xlabel='spectroscopic z', xlim=(0, 0.4), ylim=(0, 0.4))
    axs[0].set_ylabel('predicted z')
    plt.tight_layout(); plt.show()

def plot_curve():
    fig, ax = plt.subplots(figsize=(9.4, 2.9))
    ax.plot(EPOCHS, curve_tr, 'C0', label='the 400 training galaxies')
    ax.plot(EPOCHS, curve_va, 'C1', label=f'{len(y_val):,} validation galaxies (held out from training)')
    best = EPOCHS[int(np.argmin(curve_va))]
    ax.axvline(best, color='k', ls=':', lw=1); ax.text(best + 15, 0.047, f'early stopping\nwould stop here (epoch {best})', fontsize=10, va='top')
    ax.set_ylim(0, 0.05)
    ax.set(xlabel='epoch (one pass through the 400 galaxies)', ylabel='rms redshift error')
    ax.legend(fontsize=10, loc='upper right')
    plt.tight_layout(); plt.show()

def plot_wiggles(width=500, alpha=1e-6):
    curve, e_tr, e_te = fits20[width, alpha]
    fig, axs = plt.subplots(1, 2, figsize=(7.4, 2.7), gridspec_kw={'width_ratios': [3.2, 1]})
    axs[0].plot(gr[rest][:4000], z_spec[rest][:4000], ',', color='0.75')
    axs[0].plot(gr[i20], z_spec[i20], 'ko', ms=5, label='the 20 training galaxies')
    axs[0].plot(GRID, curve, 'C3', lw=2, label='network')
    axs[0].set(xlabel='g - r color', ylabel='redshift', xlim=(0.2, 1.6), ylim=(0, 0.4))
    axs[0].set_title(f'width {width}, weight decay {"none" if alpha < 1e-5 else alpha}', fontsize=10)
    axs[0].legend(fontsize=10, loc='upper center')
    axs[1].bar([0, 1], [e_tr, e_te], color=['k', '0.6'])
    axs[1].set_xticks([0, 1], ['the 20', f'other\n{len(rest):,}'], fontsize=9)
    axs[1].set(ylim=(0, 0.1), ylabel='rms error')
    for k, e in enumerate([e_tr, e_te]): axs[1].text(k, e + 0.003, f'{e:.3f}', ha='center', fontsize=9)
    plt.tight_layout(); plt.show()

def wiggles_controls():
    controls_right(plot_wiggles, width=SelectionSlider(options=WIDTHS, value=500, description='width'),
                   alpha=SelectionSlider(options=[('none', 1e-6), ('0.003', 3e-3), ('0.01', 1e-2), ('0.03', 3e-2)], value=1e-6, description='weight decay'))

def plot_shift():
    fig, ax = plt.subplots(figsize=(9.4, 2.4))
    bins = np.linspace(0, 0.45, 46)
    ax.hist(y_tr[near], bins, density=True, color='0.8', label='training galaxies: z < 0.15')
    ax.hist(y_te[far], bins, density=True, histtype='step', color='k', lw=2, label='new galaxies: true z > 0.2')
    for k, c in zip(shift, ['C2', 'C3']):
        ax.hist(shift[k], bins, density=True, histtype='step', color=c, lw=2, label=f'{k} prediction for them')
    ax.set(xlabel='redshift', ylabel='density')
    ax.legend(fontsize=10, loc='upper right')
    plt.tight_layout(); plt.show()


## Recap - last Thursday we picked the features by hand

Sorry about Tuesday (my wifi on the road couldn't handle Zoom) - today is the neural-networks lecture, and I'll let you go a little early (1:20 pm) for the break. Lab 10 is posted today, and I'm retiring lab defenses. How did Lab 09 go?

* rare classes: at 1% kilonovae, a 99% accurate forest missed 95% of them
* Sutton's bitter lesson: methods that learn from data beat the features we choose (TabPFN, trained only on synthetic tables, is one)

In [ ]:
# last Thursday's closing figure: what we chose, and what each method learned
plot_ladder()

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Today the method learns the features too.**

</div>

## Today - neural networks

* one neuron, then many: the multi-layer perceptron
* training is maximum likelihood again, solved by gradient descent
* demo: redshifts of 20,000 SDSS galaxies from their colors, then a published CNN on JWST images
* over-fitting, and the priors that stop it
* how surveys use networks on light curves, and where networks fail

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**A neural network is just a flexible function, fit by maximum likelihood.**

</div>

## One neuron: a weighted sum, then an activation function

* inputs $x$, weights $w$, bias $b$: $s = w \cdot x + b$ (Day 9's straight line), then a non-linear activation function $g(s)$
* most networks today use the ReLU, $g(s) = \max(0, s)$
* with the sigmoid $g(s) = 1/(1+e^{-s})$ the output is a probability of class 1 - i.e. logistic regression
* on Day 1 I said you'd know what logistic regression is by the end of term - this is it

In [ ]:
# four activation functions (ReLU, max(0, s), is what most networks use today)
plot_activations()

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Without a non-linear $g$, a stack of neurons is still linear regression!**

</div>

## One neuron can't do XOR

* XOR: class 1 when exactly one input is positive. Minsky and Papert (1969) showed one neuron (a perceptron) can't learn it

In [ ]:
# one neuron with a straight boundary (left) vs a hidden layer of 4 neurons, then one output (right)
plot_xor()

* a hidden layer (neurons between input and output) bends the boundary

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Hidden layers let a network draw curved boundaries.**

</div>

## Multi-layer perceptron (MLP) = a stack of design matrices

<img src="images/day23_gn2020_mlp.png" alt="A fully connected network: three inputs x1, x2, x3 each connect to four hidden neurons, each computing a weighted sum plus a bias, which all connect to one output neuron" style="display:block;margin:0 auto;max-height:270px;max-width:100%">

* hidden layer $h = g(W_1 x + b_1)$ ($W_1$ has one row of weights per hidden neuron), output $\hat{y} = W_2 h + b_2$
* for $N$ galaxies, $h$ is a new design matrix (Day 9) with one column per hidden neuron, learned from the data - like PCA's components (Day 22), but chosen to predict the label
* this is called a **multi-layer perceptron (MLP)** (Rosenblatt's 1958 perceptron was one neuron with a step activation), hence `MLPRegressor` below

<div style="font-size:0.8em; text-align:center; color:#666;">Diagram: Federica Bianco.</div>

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**The hidden layer is a design matrix the network learns from the data.**

</div>

## Photometric redshifts

<img src="images/sdss_25m_telescope.jpg" alt="The SDSS 2.5-m telescope at Apache Point Observatory, New Mexico" style="display:block;margin:0 auto;max-height:200px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">The SDSS 2.5-m telescope (credit: SDSS).</div>

* a spectrum costs far more telescope time than an image, and Rubin will image about 20 billion galaxies
* we train on galaxies with spectra (the training set), then predict z for the ones without (TPZ, Day 21, is a forest that does this)
* our data: 20,000 SDSS DR8 galaxies with u, g, r, i, z and spectroscopic z. SDSS's main galaxy sample took spectra only of galaxies brighter than $r = 17.77$, and almost all have $z < 0.3$

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Redshift is the target and the colors are the features - photo-z is regression.**

</div>

## Galaxy colors change with redshift

In [ ]:
# left: two colors of 6,000 galaxies, colored by spectroscopic redshift (right: all 20,000 redshifts)
plot_photoz_data()

* old stars' spectra drop sharply below 4000 Angstrom (the 4000 Angstrom break), and redshift moves the break through g toward r, i.e. g - r gets redder

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Color tracks redshift, and that is what the network has to learn.**

</div>

## Universal approximation

* each ReLU (one hidden neuron) adds one kink, and with enough of them one hidden layer can follow any continuous function - this is called **universal approximation** (Cybenko 1989, Hornik 1991)
* Let's fiddle: a line plus $n$ ReLUs, fit to median redshift vs g - r of SDSS galaxies. Start at 0 (the residuals bend), then 1, 4, 16 (we space the kinks evenly; a network also learns where to put them)

In [ ]:
# the median redshift vs g - r color, fit with a line plus n ReLU kinks
kinks_controls()

<img src="images/day23_static_kinks_widget.png" alt="The kinks widget at 4 ReLUs: median redshift of SDSS galaxies in bins of g - r color (black), fit by a line plus 4 ReLU kinks (red) at the dotted positions, with the residuals of the medians below" style="max-height:330px; display:block; margin:0 auto;">

(static copy of the widget, for readers on GitHub or without running the notebook)

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**With enough ReLUs, one hidden layer follows any smooth curve.**

</div>

## Training is maximum likelihood again

* regression: Gaussian errors give $-\ln L = \frac{1}{2\sigma^2}\sum_i (y_i - \hat{y}_i)^2$ + const, the mean squared error
* classification: labels are 0 or 1, $p_i$ is the network's output, and $-\ln L = -\sum_i [y_i \ln p_i + (1-y_i)\ln(1-p_i)]$ (the cross-entropy)
* a network has thousands of weights - too many to grid (Day 6) and too slow to sample with MCMC
* the gradient of the loss points uphill, so we step the other way (Day 14's HMC used the same gradient to roll its ball)

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**The loss is the negative log-likelihood, $-\ln L$, which we've minimized since Day 6.**

</div>

## Gradient descent and the learning rate

* $w \leftarrow w - \eta \, \partial(\mathrm{loss}) / \partial w$, where $\eta$ is some constant that we'll call the **learning rate**. Let's fit $z = c_0 + c_1 (g - r)$ to all 20,000 galaxies
* in a network we get the gradients from the chain rule, one layer at a time - this is called **backpropagation** (EXTRA slide at the end)

In [ ]:
# 30 steps of gradient descent from c0 = c1 = 0 (left: the path on the loss contours, right: the loss)
descent_controls()

<img src="images/day23_static_descent_widget.png" alt="The gradient-descent widget at learning rate 0.55 on raw g - r: left, the path zigzags across a long narrow valley of the loss; right, the loss minus the best possible loss falls slowly over 30 steps" style="max-height:330px; display:block; margin:0 auto;">

(static copy of the widget, for readers on GitHub or without running the notebook)

* with raw g - r the loss is a long narrow valley: 0.05 crawls, 0.55 zigzags, 0.65 blows up. Scale g - r (mean 0, sd 1) and the same 0.65 gets to the best fit in 7 steps, and only 1.05 blows up

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Scale your inputs first, then tune the learning rate.**

</div>

## A neural network in scikit-learn

* we'll call 1.48 x the median absolute deviation of $\Delta z/(1+z)$ the **scatter** ($\Delta z = z_\mathrm{phot} - z_\mathrm{spec}$; 1.48 makes it $\sigma$ for a Gaussian)
* an **outlier** has $|\Delta z|/(1+z) > 0.05$, and the offset is the median $\Delta z/(1+z)$
* `(32, 32)` means two hidden layers of 32 neurons: 1,281 weights and biases
* `early_stopping=True` holds back 10% of the training set (a validation set) and quits when its error stops falling

In [ ]:
# the same lines work for every model in the comparison: swap the regressor
scaler = StandardScaler().fit(X_tr)
net = MLPRegressor(hidden_layer_sizes=(32, 32), early_stopping=True, max_iter=300, random_state=0)
net.fit(scaler.transform(X_tr), y_tr)
z_phot = net.predict(scaler.transform(X_te))
scores(z_phot, y_te)

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Fit on the training set, score on galaxies the network never saw.**

</div>

## Network vs. forest on photo-z

In [ ]:
# predicted vs spectroscopic redshift for the 5,000 test galaxies (the dashed line is perfect)
plot_photoz_compare()

* the forest from Day 21 does as well, and so does nearest neighbors (scatter 0.017, outliers 1.9%)
* here we chose the five features - next, a network that learns its own from images

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**With a few features you chose, random forests are a good place to start.**

</div>

## Images: convolutional neural networks (CNNs)

<img src="images/day23_deepdisc_photoz.png" alt="Three panels of photometric vs spectroscopic redshift for 298 JWST galaxies: DeepDISC on 9 NIRCam images (scatter 0.031, outlier fraction 0.050), EAZY template fits with 9 NIRCam plus extra HST and JWST bands (0.020, 0.071), and EAZY with the same 9 NIRCam bands (0.040, 0.124)" style="display:block;margin:0 auto;max-height:300px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">Merz et al. 2026, Open Journal of Astrophysics 9 (arXiv:2510.27032), Fig. 9: DeepDISC photo-z from JWST NIRCam images vs EAZY template fits.</div>

* a convolution slides a small stencil of weights (say 3 x 3 pixels) over the image, and the same few weights see every pixel
* DeepDISC (a CNN) goes from JWST images straight to redshift, with no catalog magnitudes in between
* EAZY fits template galaxy spectra to measured magnitudes. With the same 9 bands, EAZY has 12.4% outliers and DeepDISC 5.0% (outliers here mean $|\Delta z|/(1+z) > 0.15$)

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**On the same 9 bands, the network learned its own features and beat the template fit.**

</div>

## The bitter lesson

<img src="images/day23_bitter_lesson_table.png" alt="Four rows, claim then what happened: chess, Dreyfus 1965 'no chess program can play even amateur chess', then deep search in 1997; Go, Piet Hut 1997 'It may be a hundred years', then AlphaGo 2016; speech and language, Chomsky 1969 'probability of a sentence' is 'an entirely useless one', then statistics and LLMs; vision, Minsky and Papert 1969 'the extension is sterile', then deep networks in 2012" style="display:block;margin:0 auto;max-height:320px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">After Josh Speagle's SkAI colloquium (2026); R. Sutton, "The Bitter Lesson" (13 March 2019, incompleteideas.net).</div>

* Sutton: "general methods that leverage computation are ultimately the most effective" - and DeepDISC just did it on JWST images

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Each time, a general method plus computation beat the rules the experts wrote down.**

</div>

## A big network over-fits 400 galaxies

* back to our five features, with a much bigger network and far fewer galaxies
* we hold back 3,000 training galaxies as a **validation set** for choosing settings (Day 21's cross-validation, with one fold). The test set is only for the final scores

In [ ]:
# a network with two layers of 256 (67,585 weights and biases) trained on 400 galaxies, one epoch at a time
plot_curve()

* the training error keeps falling, while the validation error stops falling and then slowly creeps up

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Quitting when the validation error starts rising is called early stopping.**

</div>

## Width and weight decay on 20 galaxies

* width is the number of neurons per layer, and weight decay adds $\alpha \sum w^2$ to the loss (a penalty on big weights). That penalty is a Gaussian prior on every weight, so the fit is the MAP (the peak of the posterior). Start at width 500 with no decay, then slide the decay to 0.003, 0.01 and 0.03. Then width 20 with no decay: only the widest network chases the 20 points

In [ ]:
# two hidden layers trained on 20 galaxies (black), tested on the other 19,980 (grey)
wiggles_controls()

<img src="images/day23_static_wiggles_widget.png" alt="The width and weight-decay widget at width 500 and no decay: the network curve swings through the 20 training galaxies and far from the grey cloud of the other 19,980" style="max-height:330px; display:block; margin:0 auto;">

(static copy of the widget, for readers on GitHub or without running the notebook)

* 500 wide, no decay: rms 0.013 on the 20, 0.085 on the rest. Decay 0.003 already halves that to 0.046. Day 8's elephant again: a degree-11 polynomial through 12 points

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Weight decay keeps the weights small and the curve smooth.**

</div>

## Light curves: recurrent neural networks (RAPID)

<div style="display:flex; align-items:flex-start; gap:1.2em;">
<img src="images/day23_rapid_classification_animated.gif" alt="Animation: the g and r light curve of a simulated type Ia supernova fills in day by day (top), while RAPID's probabilities for 12 classes plus pre-explosion update below it; the type Ia probability climbs past 0.8 within about two weeks of the trigger" style="max-height:330px;">
<div>

* a **recurrent neural network (RNN)** keeps a hidden state, a summary of everything seen so far, and updates it at each time step
* RAPID puts each light curve on 3-day steps and re-classifies at every one
* trained and tested on 48,029 simulated transients in 12 classes
* AUC (Day 21) 0.95 early, 0.98 late

</div>
</div>

<div style="font-size:0.8em; text-align:center; color:#666;">Muthukrishna, Narayan, Mandel et al. 2019, PASP 131, 118002 (arXiv:1904.00014), animated version of Fig. 5: a simulated type Ia and its class probabilities vs days since the survey first flagged it.</div>

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**RAPID classifies before the light curve is complete.**

</div>

## Neither model extrapolates

In [ ]:
# train on galaxies with z < 0.15 only, then predict for test galaxies with z > 0.2
plot_shift()

* both models put the z > 0.2 galaxies near 0.14.
* spectra exist only for bright galaxies, so the training set is flux-limited (Day 19's Malmquist bias): the faint, distant galaxies are the ones it saw least. In the RAPID paper we noted that our training simulations aren't fully representative

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Your training set may not be representative - check before you trust any prediction!**

</div>

## What did we learn today?

* one neuron is logistic regression, and hidden layers learn the features
* training minimizes $-\ln L$ by gradient descent
* on five features we chose, the network tied the forest. On pixels, DeepDISC learned its own and beat the templates
* big networks over-fit: hold out a validation set, stop early, and put priors on the weights
* networks fail outside their training set
* Day 20's ABC used summary statistics we chose; a network can learn them from every simulation (neural SBI, after the break)
* after the break: large language models and AI agents

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Check a neural network the way you'd check any other fit.**

</div>

## Before you go

* no class next week (fall break, Nov 21-29), and nothing is due over the break
* Lab 10 (machine learning on real ZTF supernovae) posted today - due Wed Dec 2 by noon
* Lab 09 (ABC) was due Tuesday at Noon
* I'm retiring lab defenses - they no longer count toward your grade (the syllabus is updated)
* Quiz 6 is probably Tue Dec 1
* capstones: the first ten lightning talks are Thu Dec 3 in class, and notebooks and videos are due Mon Dec 7 by Noon
* every date is in `ASSIGNMENTS.md`

## EXTRA: Rosenblatt's perceptron (1958)

<img src="images/day23_mark1_perceptron_1960.png" alt="Diagram of the Mark I Perceptron: a retina of photocells wired at random to association units, which feed eight response units that each output 1 or 0" style="display:block;margin:0 auto;max-height:320px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">Fig. 1 of the <i>Mark I Perceptron Operators' Manual</i> (Cornell Aeronautical Laboratory, 1960). Rosenblatt 1958, Psychological Review 65, 386.</div>

* a perceptron is one neuron with a step activation: output 1 or 0
* learning rule, one example at a time: $w \leftarrow w + \eta\,(t - y)\,x$, with $t$ the true label and $y$ the output
* the Mark I: a camera "retina" of photocells, random wiring, and weights set by motors on potentiometers

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**The perceptron learns its weights from labelled examples.**

</div>

## EXTRA: Backpropagation is just the chain rule

* one path through a network: $s = w_1 x$, $h = g(s)$, $\hat{y} = w_2 h$, and the loss $\ell(\hat{y})$
* forward pass: compute $s$, $h$, $\hat{y}$ and keep them
* backward pass: $\frac{\partial \ell}{\partial w_2} = \frac{\partial \ell}{\partial \hat{y}}\, h$, then $\frac{\partial \ell}{\partial w_1} = \frac{\partial \ell}{\partial \hat{y}}\, w_2\, g'(s)\, x$
* $\partial \ell / \partial \hat{y}$ is computed once and reused by every layer below (PyTorch and JAX do this automatically)

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Backpropagation gets all the gradients in one backward pass.**

</div>

## EXTRA: Minibatches, epochs and Adam

* stochastic gradient descent takes each step on a random minibatch of 32-256 objects
* an epoch is one pass through the whole training set
* Adam adapts the step size for every weight (it's scikit-learn's default)
* the Day 21 scaling lesson matters here too

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Small batches give you many more (noisier) steps per epoch.**

</div>

## EXTRA: Uneven light curves: neural ODEs (SELDON)

<img src="images/day23_seldon_arch.png" alt="SELDON architecture: an irregular multiband Rubin light curve goes into a GRU-ODE encoder that evolves a hidden state between observation times, Deep Sets pools it to a latent vector, and MLP decoders rebuild the time offset, amplitude and per-band shape of the light curve" style="display:block;margin:0 auto;max-height:300px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">Wu, O'Brien, Li et al. 2026, AAAI 40 (arXiv:2603.04392), Fig. 1: SELDON.</div>

* simulated Rubin (ELAsTiCC) type Ia have a median of 14 points over all six bands
* SELDON's encoder turns each light curve into a few numbers, and its decoder rebuilds a smooth light curve from them
* its encoder evolves the hidden state between uneven times with an ODE (no regular grid needed, like the GP on Day 17)

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**The architecture follows the data you have (images, light curves, uneven time series).**

</div>

## EXTRA: Regularization is just a prior on the weights

* weight decay is a Gaussian prior on every weight, and the fit is the MAP (the peak of the posterior)
* early stopping quits when the validation error turns up
* dropout zeroes a random half of the neurons at each step (so no single neuron matters too much)
* augmentation rotates, flips or adds noise to the training images, so the network learns what doesn't matter

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**We're adding extra information (priors) to keep the network from over-fitting.**

</div>

## EXTRA: Try it - TensorFlow Playground

* [playground.tensorflow.org](https://playground.tensorflow.org) trains a small network in your browser, and draws each neuron's output as it learns
* pick the XOR data (top right of the four), remove every hidden layer, and press play: one neuron draws one straight line
* add one hidden layer of 4 neurons: the boundary bends, as on the XOR slide
* then try the spiral with ReLU, and slide the learning rate up until it blows up

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**You can change the width, the activation and the learning rate in your browser.**

</div>

## EXTRA: How many weights does a convolution need?

* a fully connected layer of 50 neurons on a 256 x 256 color image: 196,608 x 50, about 10 million weights
* a 3 x 3 convolution with 64 filters on the same image: 3 x 3 pixels x 3 colors x 64 filters = 1,728 weights, whatever the image size
* the same few weights slide over every pixel, so a feature (an edge, a spiral arm) is found wherever it is

<div style="font-size:0.8em; text-align:center; color:#666;">Count from B. Naul, LSST DSFP Session 2 (2017), IntroToNeuralNetworks.</div>

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**A convolution shares its weights across the image, so it needs thousands of times fewer.**

</div>

## EXTRA: Trained on simulations, tested on the sky

<img src="images/day23_ciprijanovic_domain_shift.png" alt="Slide from A. Ciprijanovic: a CNN trained on simulated Illustris galaxy images (source) and tested on SDSS images (target). In the network's feature space the two sets sit apart; accuracy is above 80% on the simulations and about 50% on SDSS" style="display:block;margin:0 auto;max-height:330px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">Slide 52 of A. Ćiprijanović (Fermilab), LSST DSFP Session 25 (2026, MIT licence); figures from Ćiprijanović et al. 2020, 2021.</div>

* a CNN trained to find galaxy mergers in simulated (Illustris) images is right >80% of the time on simulations, and ~50% on real SDSS images - a coin flip
* in the network's own features (left), the simulated and real galaxies sit in two separate clouds
* domain adaptation trains the network to pull the two clouds together (Ćiprijanović et al. 2020, 2021)

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**A network trained on simulations can fail on real data - the same lesson as our redshift shift.**

</div>

## EXTRA: Why is the lesson bitter?

<img src="images/day23_speagle_weatherbench.png" alt="Five-day forecast error of learned weather models divided by that of ECMWF's physics model on the same days, by the year each model appeared: about 1.5 to 2.3 in 2020-2021, crossing below 1 from 2022-2023 onward; ECMWF switched on its own learned model in 2025" style="display:block;margin:0 auto;max-height:300px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">Figure: Josh Speagle (Toronto), SkAI colloquium (2026); WeatherBench: Rasp et al. 2020, 2024; ECMWF 2025.</div>

* building in what we know "always helps in the short term, and is personally satisfying" (Sutton)
* "in the long run it plateaus and even inhibits further progress": learned weather models were up to 2.3x worse than physics in 2020, and better from 2022 on
* ECMWF, which runs the physics model, switched on its own learned model in 2025

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Our expertise helps first, which is why it is hard to let go of.**

</div>